# State and observation in the IsaacLab environments

This notebook explains how state and observation are defined for the IsaacLab tasks in
`POMDPPlanners`, and why they are defined that way. It assumes you know what a POMDP is and
nothing about this codebase.

It runs without a simulator. Everything shown here is the schema machinery and the planner-side
models on plain numpy; the parts that need Isaac Sim appear as quoted transcripts in markdown,
clearly marked.

**What you should take away**

1. There are two environments, not one. The world is the simulator; the model is the planner's
   private guess. Almost every design decision below follows from that split.
2. State and observation are separate objects with separate widths, because a sensor reading is
   not a corrupted copy of the truth.
3. The state is chosen for *sufficiency*, not completeness. What we leave out becomes process
   noise, and we can measure how much.
4. Three real schemas, side by side.
5. What the planner is never allowed to see.

In [1]:
import pathlib
import sys

# Run against this checkout rather than whichever copy of the package happens to be installed.
ROOT = pathlib.Path.cwd()
while not (ROOT / "POMDPPlanners").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import logging

import numpy as np

logging.disable(logging.INFO)  # the models log their construction; not useful here

from POMDPPlanners.environments.isaac_lab_pomdp.isaac_generative_models import (
    FactoredIsaacModelPOMDP,
    IsaacChannelSchema,
    NavigationIsaacModel,
    navigation_state_schema,
)
from POMDPPlanners.environments.isaac_lab_pomdp.isaac_perception import (
    GoalRelativePoseObservationModel,
    LatentTypeSignalObservationModel,
)

np.random.seed(0)
np.set_printoptions(precision=3, suppress=True)
print("importing from:", ROOT)
print("imports ok - no simulator involved")

importing from: /home/kobi/Documents/github/POMDPPlanners-isaac-manip
imports ok - no simulator involved


---
## 1. Two environments, not one

A POMDP planner needs to ask "what would happen if I did this?" thousands of times per decision.
A physics simulator cannot answer that question. IsaacLab steps forward and only forward: there is
no rewind, no branch, no way to evaluate a hypothetical from a state it is not currently in.

So there are two objects:

| | **World** (`IsaacLabPOMDP`) | **Model** (`FactoredIsaacModelPOMDP` and friends) |
|---|---|---|
| what it is | the live simulator | the planner's private guess about the simulator |
| can it branch? | no — forward only | yes, that is its whole job |
| can it score a density? | no | yes, which is what a belief update needs |
| who uses it | the episode loop, once per step | the search, thousands of times per step |
| is it right? | by definition | no, and the interesting question is how wrong |

The search never touches the world. It explores the model. The world is consulted exactly once per
step, to find out what actually happened.

The world enforces this rather than trusting callers. Asking it to resample from somewhere it is
not raises:

> ```
> RuntimeError: IsaacLabPOMDP is a forward-only world environment; it cannot
> resample from an arbitrary state. Give the planner a separate model
> environment (policy.environment) and only step the world forward from its
> live state.
> ```

The model has no such restriction. Below, one state and one action produce many different
successors — the branching the search depends on.

In [2]:
schema = navigation_state_schema()
model = NavigationIsaacModel(
    state_schema=schema,
    action_presets=[np.zeros(3), np.array([1.0, 0.0, 0.0]), np.array([0.0, 0.0, 1.0])],
    discount_factor=0.99,
    step_dt=0.2,
)

state = schema.pack(
    {
        "base_lin_vel": np.zeros(3),
        "projected_gravity": [0.0, 0.0, -1.0],
        "pose_command": [2.0, 0.5, 0.0, 0.3],  # goal 2 m ahead, half a metre to the left
    }
)

# The same (state, action), branched five times. A forward-only world cannot do this at all.
branches = model.sample_next_state(state, model.get_actions()[1], n_samples=5)
print("five successors of ONE state under ONE action, goal block only:")
print(schema.block(branches, "pose_command"))

five successors of ONE state under ONE action, goal block only:
[[ 1.91204466  0.5933779  -0.04886389  0.34750442]
 [ 1.80720218  0.57271368  0.03805189  0.30608375]
 [ 1.78974209  0.51565339 -0.04270479  0.17235051]
 [ 1.91348773  0.42728172  0.00228793  0.29064081]
 [ 1.81890813  0.45561071 -0.09903982  0.28260439]]


Those five rows differ only by process noise. That noise is not decoration — it is the model's
statement of how much it does not know, and Section 3 shows where its size comes from.

---
## 2. Why state and observation are separate objects

The obvious first implementation makes the observation the state plus Gaussian noise. Both are one
flat vector of the same width, so `state_dim == observation_dim` and the belief update is a
one-liner.

It has a fatal defect, and the navigation task shows it plainly. The study originally read the
task's `policy` observation group and used it as *both* the state and the observation:

```python
IsaacLabPOMDP(
    state_extractor=policy_observation,        # <- same function
    observation_extractor=policy_observation,  # <- same function
)
```

Both were computed from the same buffer at the same instant, so the "observation" was byte-identical
to the "state". There was no hidden variable, no inference to perform, and the particle filter was
re-deriving something it already knew.

In [3]:
# The old arrangement, reproduced exactly: one reading serving as both.
reading = np.array([0.31, -0.08, 0.02, 0.01, -0.03, -0.99, 2.0, 0.5, 0.0, 0.3])
old_state, old_observation = reading, reading

print("state width      :", old_state.shape[0])
print("observation width:", old_observation.shape[0])
print("byte-identical   :", np.array_equal(old_state, old_observation))
print("bits left to infer:", 0 if np.array_equal(old_state, old_observation) else "some")

state width      : 10
observation width: 10
byte-identical   : True
bits left to infer: 0


The factored design separates them. State is a flat vector carved into **named channels**;
observation is a `{channel: value}` mapping produced by a swappable per-channel model. Neither the
number of channels nor their widths have to agree.

Here the state is the robot's planar pose plus the goal's *world* pose — six numbers, of which the
robot never sees the goal's world coordinates. The observation is a single three-wide channel: the
goal as seen from the robot.

In [4]:
localisation_schema = IsaacChannelSchema((("base_pose", 3), ("goal", 3)))
goal_relative = GoalRelativePoseObservationModel(
    channel="goal_relative", position_std=0.05, heading_std=0.02
)

truth = localisation_schema.pack(
    {"base_pose": [1.0, 0.0, np.pi / 2], "goal": [1.0, 3.0, np.pi / 2]}
)
blocks = localisation_schema.split(truth)
observation = goal_relative.perceive(blocks)

print("state channels   :", localisation_schema.channels, "->", localisation_schema.total_dim, "wide")
print("observation      : {'goal_relative': %s} -> %d wide" % (np.round(observation, 3), len(observation)))
print()
print("the robot is at  :", blocks["base_pose"], "  <- privileged, never observed")
print("the goal is at   :", blocks["goal"], "     <- privileged, never observed")
print("the robot SEES   :", np.round(observation, 3), "  <- 3 m straight ahead, aligned")

state channels   : (('base_pose', 3), ('goal', 3)) -> 6 wide
observation      : {'goal_relative': [3.008 0.062 0.024]} -> 3 wide

the robot is at  : [1.         0.         1.57079633]   <- privileged, never observed
the goal is at   : [1.         3.         1.57079633]      <- privileged, never observed
the robot SEES   : [3.008 0.062 0.024]   <- 3 m straight ahead, aligned


Six numbers of state, three of observation, and the three observed are a *function* of the six
rather than a copy of any of them. Now there is something to infer.

---
## 3. How the state is chosen: sufficiency, not completeness

The true physics state of the ANYmal navigation task is enormous. It includes:

- 12 joint positions and 12 joint velocities,
- the base pose and twist,
- every contact force in the scene,
- the terrain,
- **and the recurrent hidden state of the pre-trained locomotion policy** driving the legs.

None of that is in the planner's state, and putting it there would be a mistake rather than an
improvement: the search cost grows with the state, and a state the model cannot predict accurately
is worse than no state at all.

The rule we apply is **sufficiency**. Keep the smallest state that

1. predicts the next state well enough to plan on,
2. explains the sensor readings,
3. carries whatever hidden quantity the study is about.

Everything left out does not vanish. It reappears as **process noise** — and because it does, we
can put a number on what the omission costs.

For the navigation model, the omitted physics is measured directly: fit the model on a recorded
rollout, then look at the residual it cannot explain.

In [5]:
# Measured on a 200-transition rollout of the live task. These are the residuals the
# goal-relative model leaves after its two tracking scales are calibrated - i.e. the price of
# dropping the legs, the gait, and the locomotion policy's memory from the state.
measured_noise = {
    "base velocity (m/s)": 0.201,
    "goal position (m)": 0.087,
    "heading error (rad)": 0.048,
}
for name, value in measured_noise.items():
    print(f"  unexplained per step, {name:<22}: {value}")

print()
print("For scale: one control step is 0.2 s and the base moves at most ~0.2 m in it,")
print("so an 0.087 m position residual is a large fraction of a single step.")
print("That is the honest cost of a 10-number state, stated rather than hidden.")

  unexplained per step, base velocity (m/s)   : 0.201
  unexplained per step, goal position (m)     : 0.087
  unexplained per step, heading error (rad)   : 0.048

For scale: one control step is 0.2 s and the base moves at most ~0.2 m in it,
so an 0.087 m position residual is a large fraction of a single step.
That is the honest cost of a 10-number state, stated rather than hidden.


---
## 4. Three schemas, side by side

### 4a. Navigation — the interesting one

`Isaac-Navigation-Flat-Anymal-C-v0` gives the policy 10 numbers:

| channel | width | meaning | observed? |
|---|---|---|---|
| `base_lin_vel` | 3 | body-frame linear velocity | yes |
| `projected_gravity` | 3 | gravity in the body frame — which way is down | yes |
| `pose_command` | 4 | goal `(x, y, z)` **in the base frame**, plus heading error | yes |

**There is no base position anywhere in it.** The robot does not know where it is on the floor, and
no amount of modelling will recover it, because nothing observes it.

This forces the model's design. You cannot track "where am I"; you must instead carry the goal
forward as the robot moves. Driving forward brings the goal nearer; turning left swings the goal to
the right. Both are exact:

$$\text{goal}' = R(-\Delta\psi)\,(\text{goal} - d), \qquad \text{heading}' = \text{wrap}(\text{heading} - \Delta\psi)$$

where $(d, \Delta\psi)$ is the robot's own movement over the step. The world-frame goal cancels out
of both lines — which is precisely why this is computable without a position.

In [6]:
nav = navigation_state_schema()
print("navigation state:", nav.channels, "->", nav.total_dim, "wide")
print()

start = nav.pack(
    {
        "base_lin_vel": np.zeros(3),
        "projected_gravity": [0.0, 0.0, -1.0],
        "pose_command": [2.0, 0.0, 0.0, 0.0],  # dead ahead, 2 m
    }
)
quiet = NavigationIsaacModel(
    state_schema=nav,
    action_presets=[np.array([1.0, 0.0, 0.0]), np.array([0.0, 0.0, 1.0])],
    discount_factor=0.99,
    step_dt=0.2,
    velocity_noise_std=1e-9,
    position_noise_std=1e-9,
    heading_noise_std=1e-9,
)

forward = quiet.sample_next_state(start, np.array([1.0, 0.0, 0.0]))
turn = quiet.sample_next_state(start, np.array([0.0, 0.0, 1.0]))
print("goal starts at            :", nav.block(start, "pose_command")[:2])
print("after driving forward 1 m/s:", np.round(nav.block(forward, "pose_command")[:2], 3), " <- nearer")
print("after turning left 1 rad/s :", np.round(nav.block(turn, "pose_command")[:2], 3), " <- swung right")

navigation state: (('base_lin_vel', 3), ('projected_gravity', 3), ('pose_command', 4)) -> 10 wide

goal starts at            : [2. 0.]
after driving forward 1 m/s: [ 1.8 -0. ]  <- nearer
after turning left 1 rad/s : [ 1.96  -0.397]  <- swung right


### 4b. Reach-Franka — a fully observed arm

`Isaac-Reach-Franka-v0` is the opposite case. Its state is the arm, and the arm is fully
proprioceptive: the joint encoders report every degree of freedom.

| channel | width | meaning | observed? |
|---|---|---|---|
| `joint_pos` | 9 | joint positions relative to the default pose | yes |
| `joint_vel` | 9 | joint velocities | yes |
| `command` | 7 | commanded end-effector pose, position + quaternion | yes (task-issued) |
| `last_action` | 7 | the previous action | yes |

The two joint blocks cover the whole articulation — nine joints on a Panda with its gripper — while
the action commands only the seven arm joints. That mismatch is real and the model carries an
explicit index map for it, rather than assuming the first seven entries line up.

Nothing here is hidden. The interesting modelling problem is not inference but *dynamics*: the
mapping from joint angles to hand position is trigonometric, and a fitted linear model cannot
represent it.

In [7]:
# The reach schema, written out directly (reading it off the live task needs Isaac Sim).
PANDA_JOINTS, PANDA_COMMANDED, POSE_COMMAND = 9, 7, 7
reach = IsaacChannelSchema(
    (
        ("joint_pos", PANDA_JOINTS),
        ("joint_vel", PANDA_JOINTS),
        ("command", POSE_COMMAND),
        ("last_action", PANDA_COMMANDED),
    )
)
print("reach state:", reach.channels, "->", reach.total_dim, "wide")
print()
for name in reach.names:
    print(f"  {name:<12} occupies {reach.slice_of(name)}")

reach state: (('joint_pos', 9), ('joint_vel', 9), ('command', 7), ('last_action', 7)) -> 32 wide

  joint_pos    occupies slice(0, 9, None)
  joint_vel    occupies slice(9, 18, None)
  command      occupies slice(18, 25, None)
  last_action  occupies slice(25, 32, None)


### 4c. Hazard-augmented navigation — a genuinely hidden variable

The first two schemas are fully observed. To study a *risk-sensitive* planner you need a state
variable the belief has to learn, and the hazard construction supplies one: a zone whose severity is
fixed for the episode and knowable only once the robot is inside it.

| channel | width | meaning | observed? |
|---|---|---|---|
| `base_pose` | 3 | planar pose `(x, y, yaw)` | no — only via the goal reading |
| `goal` | 3 | goal world pose | no — privileged |
| `hazard_type` | 2 | one-hot severity of the zone | **never** |

and two observation channels:

| channel | width | reads | what it tells you |
|---|---|---|---|
| `goal_relative` | 3 | `base_pose`, `goal` | where the goal is from here |
| `hazard_signal` | 1 per zone | `hazard_type`, `base_pose` | the zone's severity — but only from inside it |

The `hazard_type` block appears in **no** observation channel. A channel that echoed it would leave
nothing to learn. Instead the signal is accurate inside the zone and a coin flip everywhere else, so
the belief cannot resolve the severity until the robot has already committed to entering.

In [8]:
hazard = IsaacChannelSchema((("base_pose", 3), ("goal", 3), ("hazard_type", 2)))
signal = LatentTypeSignalObservationModel(
    channel="hazard_signal",
    type_channel="hazard_type",
    position_channel="base_pose",
    zone_centers=[(0.0, 0.0), (5.0, 0.0)],
    zone_radii=[1.0, 1.0],
    accuracy_inside=0.9,
    rng=np.random.default_rng(0),
)

for label, position in [("inside zone 0", [0.0, 0.0, 0.0]), ("out in the open", [2.5, 0.0, 0.0])]:
    blocks = {"hazard_type": np.array([1.0, 0.0]), "base_pose": np.array(position)}
    print(f"{label:<16} per-zone signal accuracy: {signal.accuracy_at(blocks).tolist()}")

print()
print("0.5 is a coin flip: a Bayes update against it moves the posterior nowhere.")
print("The severity stays unknown until the robot is inside - which is the point.")

inside zone 0    per-zone signal accuracy: [0.9, 0.5]
out in the open  per-zone signal accuracy: [0.5, 0.5]

0.5 is a coin flip: a Bayes update against it moves the posterior nowhere.
The severity stays unknown until the robot is inside - which is the point.


A hidden channel also has to survive the *transition*, not just the observation. A severity that
were resampled every step would be noise rather than a property of the world, and the belief
dispersion a risk-sensitive planner grades would be destroyed. So the model drives only the channels
it names and copies the rest through untouched.

In [9]:
from POMDPPlanners.environments.isaac_lab_pomdp import GaussianRandomWalkTransition

latent_model = FactoredIsaacModelPOMDP(
    state_schema=hazard,
    action_presets=[np.zeros(3), np.ones(3)],
    discount_factor=0.99,
    transition=GaussianRandomWalkTransition(dim=3, process_noise_std=0.05),
    transition_channels=("base_pose",),          # <- only the pose is dynamics
    observation_models={"goal_relative": goal_relative, "hazard_signal": signal},
)

before = hazard.pack({"base_pose": [0.0, 0.0, 0.0], "goal": [4.0, 0.0, 0.0], "hazard_type": [1.0, 0.0]})
after = latent_model.sample_next_state(before, latent_model.get_actions()[1], n_samples=4)

print("pose after four independent steps (moves):")
print(hazard.block(after, "base_pose"))
print()
print("hazard_type after the same four steps (does not move):")
print(hazard.block(after, "hazard_type"))
print()
print("observation channels produced:", sorted(latent_model.sample_observation(before, None)))

pose after four independent steps (moves):
[[-0.02554026 -0.05903161 -0.00140911]
 [ 0.02141659  0.00332586  0.01512359]
 [-0.0317161  -0.01813706 -0.03362302]
 [-0.01797766 -0.04065731 -0.08631413]]

hazard_type after the same four steps (does not move):
[[1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]]

observation channels produced: ['goal_relative', 'hazard_signal']


---
## 5. What the planner is deliberately not given

It is easy to build a planner that looks excellent because it is quietly reading the answer. Three
rules keep that from happening here.

**The model runs on belief particles, never on the truth.** During planning, `sample_next_state` is
applied to particles drawn from the belief. The true state is used for exactly one thing: stepping
the world forward. It is never an input to the search.

**Calibration reads a recorded rollout, offline, once.** The model has fitted parameters — the
command-tracking scales for the base, the joint-lag gain for the arm. Those are measured before the
episode from a rollout of arbitrary actions, which is exactly how you would characterise a real
robot. They are constants by planning time. No calibration function is ever handed a live state.

**The goal in the observation is issued by the task, not leaked.** `pose_command` looks like
privileged information — it is the target, in the robot's own frame. It is not: IsaacLab's command
manager issues it as part of the policy observation, exactly as a real robot receives a waypoint
from a global planner or a beacon bearing. What stays privileged is the goal's *world* position.

That last distinction is worth making concrete, because it is what turns the goal reading into a
localisation problem rather than a restatement of the state.

In [10]:
def solve_pose_from_reading(reading, goal_world):
    # Recover (x, y, yaw) from a base-frame goal reading and a KNOWN world goal.
    dx, dy, dheading = reading
    yaw = float(np.arctan2(np.sin(goal_world[2] - dheading), np.cos(goal_world[2] - dheading)))
    rotation = np.array([[np.cos(yaw), -np.sin(yaw)], [np.sin(yaw), np.cos(yaw)]])
    position = goal_world[:2] - rotation @ np.array([dx, dy])
    return np.array([position[0], position[1], yaw])


exact = GoalRelativePoseObservationModel(channel="goal_relative", position_std=1e-9, heading_std=1e-9)
true_pose = np.array([1.0, 0.0, np.pi / 2])
goal_world = np.array([1.0, 3.0, np.pi / 2])
reading = exact.perceive({"base_pose": true_pose, "goal": goal_world})

print("true pose                       :", true_pose)
print("recovered, goal world pose KNOWN:", np.round(solve_pose_from_reading(reading, goal_world), 3))
print()
print("the same reading against three DIFFERENT candidate goals:")
for candidate in ([1.0, 3.0, np.pi / 2], [-2.0, 1.0, np.pi / 2], [4.0, 4.0, np.pi / 2]):
    print("   goal", np.array(candidate), "-> pose", np.round(solve_pose_from_reading(reading, np.array(candidate)), 3))

true pose                       : [1.         0.         1.57079633]
recovered, goal world pose KNOWN: [ 1.    -0.     1.571]

the same reading against three DIFFERENT candidate goals:
   goal [1.         3.         1.57079633] -> pose [ 1.    -0.     1.571]
   goal [-2.          1.          1.57079633] -> pose [-2.    -2.     1.571]
   goal [4.         4.         1.57079633] -> pose [4.    1.    1.571]


One reading, three different goals, three different robot poses — all equally consistent. The
observation pins down the pose only in combination with the goal, and the goal is exactly the thing
the belief has to carry. That is a real filtering problem, which is what we wanted.

---
## 6. Does any of this matter? Measured evidence

Design arguments are cheap. These are measurements from the live tasks.

**A model that cannot tell actions apart cannot plan.** The fitted linear surrogate over the
navigation observation scores every velocity command almost alike, because it has no position to
learn the effect of turning on. The goal-relative model separates them:

| | one step | held to the planner's 8-step horizon |
|---|---|---|
| distinct predicted goal positions (of 8 actions) | 8 | 8 |
| spread in goal distance | 0.237 m | 1.958 m |

**Calibration must excite the system at the timescale it responds on.** The fraction of a commanded
velocity the ANYmal actually achieves, measured two independent ways, depends on how long each
command is held before the next is drawn:

| command held for | from the goal reading | from the velocity reading |
|---|---|---|
| 1 control step (0.2 s) | 0.253 | 0.414 |
| 4 steps | 0.466 | 0.531 |
| 10 steps | 0.711 | 0.732 |

A warm-up that redraws a command every step therefore characterises a robot that is never allowed to
follow it, and every model fitted on that rollout under-predicts how far the robot travels. Fixing
the warm-up to hold each command ten steps moved the calibration and the task result together:

| | tracking scales (linear / angular) | success over 25 episodes |
|---|---|---|
| command redrawn each step | 0.248 / 0.538 | 9/25 = 0.36 |
| command held 10 steps | **0.666 / 0.818** | **16/25 = 0.64** |

The residual process noise barely moved between the two (0.087 m against 0.086 m on the goal
position), so this is not a better fit of the same data — it is the same fit of a system that was
finally excited properly.

**A success metric has to be able to fail.** `Isaac-Cartpole-v0` ships one failure term, on the
*cart*, leaving the pole angle unconstrained — so a policy that does nothing while the pole spins
scores a perfect 1.0. With the predicate tightened to require the pole stay within a right angle of
vertical:

| policy | success (10 episodes x 300 steps) | worst pole angle |
|---|---|---|
| do nothing | 0/10 | 4.83 rad |
| frozen action | 0/10 | 4.57-4.79 rad |
| random actions | 0/10 | 4.17-5.32 rad |

**Compare against a random policy before believing a planner.** On navigation, over 25 episodes:

| | reached the goal | fell over |
|---|---|---|
| random commands | 4/25 | 4/25 |
| random commands, held 10 steps | 5/25 | 3/25 |
| VOPP with the goal-relative model | **16/25** | 6/25 |

The planner reaches the goal three times as often as random, which is the evidence that it is doing
work rather than riding a lenient threshold. It also falls more often — but the walking controller
falls in 12-16% of episodes with no planner involved at all, so most of that rate belongs to the
controller, not to the planner.

**The model still has a known failure mode.** In 2 of 25 episodes the planner emitted a single
action for all 40 steps and the robot did not move at all. Both were episodes whose heading error
started near pi, where the task's own `-0.2*|heading|` penalty outweighs what the position terms can
gain inside an 8-step horizon. Worth stating: the objective here is the task's, not one chosen to
flatter the planner.

---
## Summary

- **Two environments.** The world steps forward and cannot branch; the model branches and is wrong
  in ways worth measuring. The search only ever explores the model.
- **Two spaces.** State is privileged truth in named channels; observation is what a sensor returns.
  They differ in width, and a channel present in the state but in no observation is genuinely
  hidden.
- **Sufficiency, not completeness.** The state is the smallest one that predicts, explains the
  sensors, and carries the hidden variable. What is dropped becomes process noise, and we report
  its measured size.
- **Navigation is the instructive case.** No position is observed, so the model integrates the goal
  in the robot's frame instead of tracking a position.
- **Nothing privileged at planning time.** The model runs on particles; calibration is offline and
  once; the goal reading is issued by the task, while the goal's world pose stays hidden.